#Prohlášení
Tento notebook slouží ke trénování modelů s architekturou Donut a byl vytvořen z velké části na základě noteboků Nielse Rogge, repozitář: https://github.com/NielsRogge/Transformers-Tutorials

#Proměnné prostředí


In [ ]:
!git clone https://github.com/TomasFAV/InvoiceCzech.git /content/InvoiceCzech

Cloning into '/content/InvoiceCzech'...
remote: Enumerating objects: 545, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 545 (delta 69), reused 103 (delta 38), pack-reused 398 (from 1)
Receiving objects: 100% (545/545), 40.02 MiB | 22.84 MiB/s, done.
Resolving deltas: 100% (121/121), done.


In [ ]:
!pip install zss
!pip install -q pytesseract

  Preparing metadata (setup.py) ... done
  Created wheel for zss: filename=zss-1.2.0-py3-none-any.whl size=6725 sha256=a3c9a4a6f8e223cbc02ffcb7137f5c04c4956e96bcea5fd22d873203728831d0
  Stored in directory: /root/.cache/pip/wheels/46/e7/2e/44fb39352ad468427a7528cacbefefaa438a898dfd1ad2eaa4
Successfully built zss


In [ ]:
import torch

#BASE_MODEL_ID = "naver-clova-ix/donut-base-finetuned-cord-v2"
BASE_MODEL_ID = "TomasFAV/DonutInvoiceCzechV01R"
FINETUNED_MODEL_ID = "TomasFAV/DonutInvoiceCzechV013R"

#DATASETY
TRAININGD_DATASET_ID = "TomasFAV/DocumentInvoiceCzechV3"
VALIDATION_DATASET_ID = "TomasFAV/RealDocumentInvoiceCzech"

IMAGE_SIZE = [1654, 2338]
MAX_LENGTH = 768

task_start_token = "<s_cord-v2>"
task_end_token = "</s>"
ADD_SPECIAL_TOKENS = False

LR = 9e-5
TRAIN_BATCH_SIZE = 4

IGNORE_ID = -100

device = "cuda" if torch.cuda.is_available() else "cpu"

#DATASETY

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset(TRAININGD_DATASET_ID, split="train")
val_dataset = load_dataset(VALIDATION_DATASET_ID, split="train")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/184 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/13.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39 [00:00<?, ? examples/s]

## OVERFIT
slouží k ověření, zda se model učí

In [ ]:
#x = 4
#train_dataset = train_dataset.select(range(x))
#val_dataset = val_dataset.select(range(x))
#train_dataset

#Převod JSONU => TOKENY (xml styl)


In [ ]:
new_tokens = set()
new_tokens.update([task_start_token, task_end_token])

def json2token(obj) -> str:
    # 🔹 None nebo "None" → prázdný obsah
    if obj is None:
        return ""

    if isinstance(obj, str) and obj.strip().lower() == "none":
        return ""

    if isinstance(obj, dict):
        if len(obj) == 1:
            v = next(iter(obj.values()))
            if v is None or (isinstance(v, str) and v.strip().lower() == "none"):
                return ""
            return str(v)

        output = ""
        keys = sorted(obj.keys())
        for k in keys:
            new_tokens.update([fr"<s_{k}>", fr"</s_{k}>"])
            output += (
                fr"<s_{k}>"
                + json2token(obj[k])   # None / "None" → ""
                + fr"</s_{k}>"
            )
        return output

    elif isinstance(obj, list):
        new_tokens.add("<sep/>")
        return "<sep/>".join([json2token(item) for item in obj])

    else:
        obj_str = str(obj)
        if f"<{obj_str}/>" in new_tokens:
            obj_str = f"<{obj_str}/>"
        return obj_str

In [ ]:
import json

def clean_docs_for_donut(sample):
    gt = sample["ground_truth"]
    # Check if gt is a string and try to parse it as JSON
    if isinstance(gt, str):
        gt = json.loads(gt)

    parsed_text = gt['gt_parse'] if 'gt_parse' in gt else gt['gt_parses']
    text = task_start_token+json2token(parsed_text)+task_end_token
    return {"text": text}


train_dataset = train_dataset.map(clean_docs_for_donut)
val_dataset = val_dataset.map(clean_docs_for_donut)

print(f"Sample:\n{train_dataset[4]["text"]}")

Map:   0%|          | 0/184 [00:00<?, ? examples/s]

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

Sample:
<s_cord-v2><s_bank_account_number>006007-0700103393/0300</s_bank_account_number><s_bic></s_bic><s_const_symbol></s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date>25.11.2024</s_due_date><s_iban></s_iban><s_invoice_number></s_invoice_number><s_issue_date>31.10.2024</s_issue_date><s_payment_type>inkasem</s_payment_type><s_supp_register_id>60193336</s_supp_register_id><s_supp_tax_id>CZ60193336</s_supp_tax_id><s_taxable_supply_date>31.10.2024</s_taxable_supply_date><s_total>2318,84</s_total><s_variable_symbol>5254326863</s_variable_symbol></s>


#Přednastavení modelu

In [ ]:
from transformers import VisionEncoderDecoderConfig

config = VisionEncoderDecoderConfig.from_pretrained(BASE_MODEL_ID)

config.encoder.image_size = IMAGE_SIZE
config.decoder.max_length = MAX_LENGTH

config.json: 0.00B [00:00, ?B/s]

#Načtení samotného modelu

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel



processor = DonutProcessor.from_pretrained(BASE_MODEL_ID, use_fast = True)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL_ID, config=config)

tokenizer = processor.tokenizer


processor_config.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/806M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

#Přidání značek < s_key > do slovníku modelu
Násobek 64, kvůli lepšímu výkonu, viz. https://x.com/karpathy/status/1621578354024677377?lang=en

In [ ]:
new_special_tokens = list(set(new_tokens))

print(f"New special tokens:  {new_special_tokens}")

# add new special tokens to tokenizer
print(f"Adding {len(new_special_tokens)} special tokens")
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens}) #takhle bychom se měli vyhnout duplikacím

# https://x.com/karpathy/status/1621578354024677377?lang=en
multiplier = 64
if len(processor.tokenizer) % multiplier != 0:
  extra_token_count = multiplier - (len(processor.tokenizer) % multiplier)
  print(f"Adding {extra_token_count} fake tokens")
  fake_tokens = [f"<reserved_{i+1}>" for i in range(extra_token_count)]
  processor.tokenizer.add_tokens(fake_tokens)

print(f"New tokenizer length: {len(processor.tokenizer)}")
# Resize model embeddings to match the new tokenizer size
model.decoder.resize_token_embeddings(len(processor.tokenizer))

New special tokens:  ['<s_variable_symbol>', '<s_bic>', '</s_iban>', '</s_supp_register_id>', '<s_cust_register_id>', '<s_taxable_supply_date>', '<s_const_symbol>', '</s_invoice_number>', '<s_supp_register_id>', '<s_bank_account_number>', '<s_payment_type>', '</s_bic>', '</s_variable_symbol>', '</s_due_date>', '</s_issue_date>', '</s_payment_type>', '</s_cust_register_id>', '<s_supp_tax_id>', '<s_iban>', '<s_total>', '</s_const_symbol>', '</s_taxable_supply_date>', '<s_cust_tax_id>', '<s_invoice_number>', '</s_total>', '<s_due_date>', '</s_bank_account_number>', '</s_supp_tax_id>', '<s_issue_date>', '</s_cust_tax_id>', '<s_cord-v2>', '</s>']
Adding 32 special tokens
New tokenizer length: 57664


MBartScaledWordEmbedding(57664, 1024, padding_idx=1)

#Nastavení modelu


In [ ]:
processor.image_processor.do_align_long_axis = False
processor.image_processor.size['width'] =  IMAGE_SIZE[0]
processor.image_processor.size['height'] =  IMAGE_SIZE[1]


model.encoder.image_size = IMAGE_SIZE
model.decoder.max_length = MAX_LENGTH
#model.config.decoder_start_token_id = tokenizer.convert_tokens_to_ids([task_start_token])[0]
task_token_id = processor.tokenizer.convert_tokens_to_ids(task_start_token)

model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

model.generation_config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id = processor.tokenizer.eos_token_id

model.config.encoder.image_size = IMAGE_SIZE[::-1]


# COLLATOR

In [ ]:
def collator(batch):
    images = [item["image"].convert("RGB").resize(IMAGE_SIZE) for item in batch]
    texts = [item["text"] for item in batch]
    inputs = processor.image_processor(
        images=images,
        return_tensors="pt",
    )


    # Příprava labelů (text, který se má model naučit generovat)
    # Musíme tokenizovat cílový text
    labels_inputs = processor.tokenizer(text=texts, padding="max_length", return_tensors="pt", truncation=True, add_special_tokens=ADD_SPECIAL_TOKENS, max_length=MAX_LENGTH)
    labels = labels_inputs.input_ids.clone()

    # Maskování paddingu pro CrossEntropyLoss
    labels[labels == processor.tokenizer.pad_token_id] = IGNORE_ID #-100 je ignore_id

    inputs["labels"] = labels

    return inputs

#Testovaní collator funkce

In [ ]:
# TEST
batch= collator([
    train_dataset[1],
])
batch.keys()

KeysView({'pixel_values': tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]]), 'labels': tensor([[57579, 57580, 57581, 57598, 57599, 57594, 57605, 57604, 57588, 57584,
         57602, 57607, 57586, 57596, 57603, 57589, 22081, 27842, 56548,  818

In [ ]:
pixel_values = batch["pixel_values"]
labels = batch["labels"]

##Převod zpět na text pro kontrolu

In [ ]:
clean_labels = labels[labels != -100]
for t in clean_labels.tolist():
    print(t, tokenizer.decode(t))
print(tokenizer.decode(clean_labels))

57579 <s_cord-v2>
57580 <s_bank_account_number>
57581 </s_bank_account_number>
57598 <s_bic>
57599 </s_bic>
57594 <s_const_symbol>
57605 </s_const_symbol>
57604 <s_cust_register_id>
57588 </s_cust_register_id>
57584 <s_cust_tax_id>
57602 </s_cust_tax_id>
57607 <s_due_date>
57586 </s_due_date>
57596 <s_iban>
57603 </s_iban>
57589 <s_invoice_number>
22081 4
27842 24
56548 10
8189 47
18980 997
57587 </s_invoice_number>
57606 <s_issue_date>
41796 02
39539 .
34585 09.
12965 20
27842 24
57592 </s_issue_date>
57591 <s_payment_type>
40345 Kart
52013 ou
3001 online
57600 </s_payment_type>
57582 <s_supp_register_id>
49617 25
46550 220
14153 68
38167 3
57595 </s_supp_register_id>
57601 <s_supp_tax_id>
42990 
43699 CZ
34173 25
46550 220
14153 68
38167 3
57590 </s_supp_tax_id>
57583 <s_taxable_supply_date>
57593 </s_taxable_supply_date>
57544 <s_total>
50090 17
6129 00,00
57543 </s_total>
57597 <s_variable_symbol>
57585 </s_variable_symbol>
2 </s>
<s_cord-v2><s_bank_account_number></s_bank_account_

#METRIKY A POMOCNÉ VĚCI PRO VALIDACI

In [ ]:
from InvoiceCzech.evaluation_utils.utils import *

In [ ]:
from torch.utils.data import DataLoader

val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,        # Nastavte podle paměti GPU
    shuffle=False,
    collate_fn=collator, # Vaše funkce, kterou jsme ladili
    num_workers=0        # Pro začátek nechte 0, aby se lépe debugovaly chyby
)

In [ ]:
def clean_text(x):
    x = x.replace(processor.tokenizer.pad_token, "")
    x = x.replace(processor.tokenizer.eos_token, "")
    return x.strip()

In [ ]:
import numpy as np
#from donut import JSONParseEvaluator

def compute_metrics_local(eval_pred):
    predictions, labels = eval_pred

    # Pokud model vrací tuple, vezmeme první prvek
    if isinstance(predictions, tuple):
        predictions = predictions[0]


    # U labels nahradíme -100 za pad_token_id, aby šly dekódovat
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    predictions = np.where(predictions != -100, predictions, processor.tokenizer.pad_token_id)

    # Decode do stringů
    pred_str = [clean_text(x) for x in processor.batch_decode(predictions, skip_special_tokens=False)]
    label_str = [clean_text(x) for x in processor.batch_decode(labels, skip_special_tokens=False)]

    preds, answ = [],[]
    for pred, label in zip(pred_str, label_str):
        pred_json = processor.token2json(pred)
        label_json = processor.token2json(label)


        preds.append(pred_json)
        answ.append(label_json)


    metrics = compute_metrics(preds, answ)
    return {
        "accuracy": metrics["accuracy"],
        "f1": metrics["macro-f1"]
    }

#Vizualizační callback
Umožňuje dělat zobrazení toho, co se model zatím naučil generovat

In [ ]:
import torch
from transformers import TrainerCallback

class VisualProgressCallback(TrainerCallback):
    def __init__(self, eval_dataloader, processor, device):
        self.eval_dataloader = eval_dataloader
        self.processor = processor
        self.device = device

    def on_evaluate(self, args, state, control, model, **kwargs):
        model.eval()

        # 1. Získáme batch přímo z eval_dataloaderu (to je nejjistější)
        eval_dataloader = self.eval_dataloader
        batch = next(iter(eval_dataloader))

        # 2. Přesun na device a omezení na 1 vzorek pro rychlost
        pixel_values = batch["pixel_values"][:1].to(self.device)
        labels = batch["labels"][:1]

        print(f"\n\n--- UKÁZKA GENERACE (Epocha {state.epoch:.1f}) ---")

        try:
            with torch.no_grad():
                generated = model.generate(
                    pixel_values,
                    max_length=MAX_LENGTH,
                    early_stopping=True,
                    pad_token_id=processor.tokenizer.pad_token_id,
                    eos_token_id=processor.tokenizer.eos_token_id,
                    use_cache=True,
                    bad_words_ids=[[processor.tokenizer.unk_token_id]],
                    return_dict_in_generate=True,
                )

                prediction = processor.tokenizer.decode(
                    generated.sequences[0],
                    skip_special_tokens=False
                )

                clean_labels = [t for t in labels[0].tolist() if t != -100]
                ground_truth = processor.tokenizer.decode(
                    clean_labels,
                    skip_special_tokens=False
                )

                print(f"  GT:   {ground_truth}")
                print(f"  PRED: {prediction}")
        except Exception as e:
            print(f"  Chyba při generování: {e}")

        print("-" * 50 + "\n")

#Trénink s HUGGINGFACE API

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# 1. Definice argumentů pro trénink
training_args = Seq2SeqTrainingArguments(
    output_dir= FINETUNED_MODEL_ID,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,        # Upravte podle paměti GPU
    per_device_eval_batch_size=1,
    learning_rate=LR,                   # Vaše MAX_LR
    num_train_epochs=20,


    max_grad_norm=1.0,
    # Validace a ukládání
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=2,

    # Mixed precision (automaticky nahradí GradScaler a autocast)
    fp16=True,

    # Generování během evaluace (pro vaši vizuální kontrolu)
    predict_with_generate=True,
    generation_max_length=768,

    # Odstranění nepoužitých sloupců
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better = True,

    report_to=["tensorboard"],
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=FINETUNED_MODEL_ID,
)

from transformers import get_scheduler

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=500, num_training_steps=10000
)

# 3. Inicializace Traineru
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset, # Předpokládám, že máte Dataset objekt
    eval_dataset=val_dataset,
    # Pokud používáte vlastní dataloader, můžete použít data_collator
    data_collator=collator,
    optimizers=(optimizer, None), # Předáme naše připravené objekty, opraveno: přidáno None pro scheduler
    callbacks=[VisualProgressCallback(val_dataloader, processor, device)],
    compute_metrics = compute_metrics_local,
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.256059,0.195757,0.881925,0.858120
2,0.147927,0.183943,0.902560,0.882051
3,0.073036,0.194075,0.910646,0.888889
4,0.048196,0.208737,0.905285,0.887179
5,0.020872,0.216293,0.898422,0.869083
6,0.026983,0.223453,0.917155,0.882051
7,0.013085,0.225027,0.914185,0.886177
8,0.036112,0.224149,0.912855,0.884860
9,0.008654,0.246510,0.916992,0.890598
10,0.018765,0.235066,0.910818,0.885470


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




--- UKÁZKA GENERACE (Epocha 1.0) ---
  GT:   <s_cord-v2><s_bank_account_number> 830809001/5500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ5955000000000830809001</s_iban><s_invoice_number></s_invoice_number><s_issue_date> 24.12.2025</s_issue_date><s_payment_type></s_payment_type><s_supp_register_id> 26043319</s_supp_register_id><s_supp_tax_id> CZ26043319</s_supp_tax_id><s_taxable_supply_date></s_taxable_supply_date><s_total> 441,65</s_total><s_variable_symbol> 5251227361</s_variable_symbol></s>
  PRED: <s><s_cord-v2><s_bank_account_number> 830809001/5500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date> 31.10.2026</s_due_date><s_iban> CZ595500000000083080900</s_iban><s_invoice_number> 5251227361</s_invoice_number

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]



--- UKÁZKA GENERACE (Epocha 2.0) ---
  GT:   <s_cord-v2><s_bank_account_number> 830809001/5500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ5955000000000830809001</s_iban><s_invoice_number></s_invoice_number><s_issue_date> 24.12.2025</s_issue_date><s_payment_type></s_payment_type><s_supp_register_id> 26043319</s_supp_register_id><s_supp_tax_id> CZ26043319</s_supp_tax_id><s_taxable_supply_date></s_taxable_supply_date><s_total> 441,65</s_total><s_variable_symbol> 5251227361</s_variable_symbol></s>
  PRED: <s><s_cord-v2><s_bank_account_number> 830809001//5500</s_bank_account_number><s_bic> RZBOCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ59550000000008308090</s_iban><s_invoice_number> 5251227361</s_invoice_number>

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]



--- UKÁZKA GENERACE (Epocha 20.0) ---
  GT:   <s_cord-v2><s_bank_account_number> 830809001/5500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ5955000000000830809001</s_iban><s_invoice_number></s_invoice_number><s_issue_date> 24.12.2025</s_issue_date><s_payment_type></s_payment_type><s_supp_register_id> 26043319</s_supp_register_id><s_supp_tax_id> CZ26043319</s_supp_tax_id><s_taxable_supply_date></s_taxable_supply_date><s_total> 441,65</s_total><s_variable_symbol> 5251227361</s_variable_symbol></s>
  PRED: <s><s_cord-v2><s_bank_account_number> 8308090015500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id> CZ2643319</s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ595500000000083080900</s_iban><s_invoice_number> 5251227361</s_invoic

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.lm_head.weight'].


TrainOutput(global_step=920, training_loss=0.041134736304545935, metrics={'train_runtime': 1205.2358, 'train_samples_per_second': 3.053, 'train_steps_per_second': 0.763, 'total_flos': 3.6212811059866575e+19, 'train_loss': 0.041134736304545935, 'epoch': 20.0})

#EVALUACE
Nutné spustit, kvůli automatickému vytvoření karty modelu v huggingface hubu

In [ ]:
trainer.evaluate()



--- UKÁZKA GENERACE (Epocha 20.0) ---
  GT:   <s_cord-v2><s_bank_account_number> 830809001/5500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id></s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ5955000000000830809001</s_iban><s_invoice_number></s_invoice_number><s_issue_date> 24.12.2025</s_issue_date><s_payment_type></s_payment_type><s_supp_register_id> 26043319</s_supp_register_id><s_supp_tax_id> CZ26043319</s_supp_tax_id><s_taxable_supply_date></s_taxable_supply_date><s_total> 441,65</s_total><s_variable_symbol> 5251227361</s_variable_symbol></s>
  PRED: <s><s_cord-v2><s_bank_account_number> 8308090015500</s_bank_account_number><s_bic> RZBCCZPP</s_bic><s_const_symbol> 308</s_const_symbol><s_cust_register_id></s_cust_register_id><s_cust_tax_id> CZ2643319</s_cust_tax_id><s_due_date> 3.1.2026</s_due_date><s_iban> CZ5955000000000830809001</s_iban><s_invoice_number> 5251227361</s_invoi

{'eval_loss': 0.2354734092950821,
 'eval_accuracy': 0.9150386809250899,
 'eval_f1': 0.9042735042735043,
 'eval_runtime': 14.8111,
 'eval_samples_per_second': 2.633,
 'eval_steps_per_second': 2.633,
 'epoch': 20.0}

# PUSH TO HUB

In [ ]:
trainer.save_model(FINETUNED_MODEL_ID)
processor.save_pretrained(FINETUNED_MODEL_ID)

# Save processor and create model card
trainer.create_model_card()
trainer.push_to_hub()
processor.push_to_hub(repo_id=FINETUNED_MODEL_ID)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...chV013R/training_args.bin: 100%|##########| 5.33kB / 5.33kB            

...

  ...95987.27aa06d52b55.5780.1: 100%|##########|   457B /   457B            

  ...chV013R/model.safetensors:  47%|####6     |  376MB /  806MB            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/TomasFAV/DonutInvoiceCzechV013R/commit/0ee7a5728ae4ebd414fdabde2997007f44cb64f6', commit_message='Upload processor', commit_description='', oid='0ee7a5728ae4ebd414fdabde2997007f44cb64f6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/TomasFAV/DonutInvoiceCzechV013R', endpoint='https://huggingface.co', repo_type='model', repo_id='TomasFAV/DonutInvoiceCzechV013R'), pr_revision=None, pr_num=None)

In [ ]:
from google.colab import runtime
runtime.unassign()